# Pierwsza paczka klienta — jakość, uderzenia i porównanie nagrań
**Run All** przelicza lokalne nagrania narzędziami `pinglab`, generuje wykresy, CSV i raport `START.html`. Wybierz kernel `.venv` / Ping Lab. Dane są w `data/raw/client-20260929/Recordings` (prywatna kopia ZIP; oryginały bez zmian).

Bad 1–2 traktujemy roboczo jako dwie pęknięte płytki. Bitten analizujemy osobno. ID płytek w OLD są nieznane. Nie dzielimy losowo uderzeń na trening i test i nie raportujemy accuracy bez poprawnego podziału po fizycznych płytkach.

In [ ]:
from pathlib import Path
import sys
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/pinglab').is_dir())
sys.path.insert(0,str(ROOT/'src'));sys.path.insert(0,str(ROOT/'scripts'))
from IPython.display import display, Markdown, Image, Audio
import pandas as pd
import numpy as np
import soundfile as sf
from analyze_client_recordings import run, OUT
SOURCE=ROOT/'data/raw/client-20260929/Recordings'
RUN_AST=True # lokalne wagi i cache, bez pobierania; ustaw False dla szybszego przeglądu
records, events=run(SOURCE)
print(f'{len(records)} pliki | {records.duration_s.sum()/60:.1f} min | {len(events)} kandydatów do cech')

## 1. Co faktycznie dostaliśmy?
Jednostką oceny jest fizyczna płytka. Licznik zdarzeń nie oznacza liczby niezależnych przykładów. Kolumna `plate_id` dla Bad pochodzi z folderu i wymaga potwierdzenia; dla OLD pozostaje pusta.

In [ ]:
display(records.groupby(['folder','label']).agg(pliki=('recording_id','size'),sekundy=('duration_s','sum'),kandydaci_do_cech=('n_feature_candidates','sum')).round(1))
display(records[['recording_id','path','native_sr','codec','duration_s','peak_dbfs','clipping_fraction','zero_fraction','channel_max_difference']].round(4))

## 2. Nagranie pod lupą — zmień RECORDING_ID
Wszystkie pliki są 48 kHz; kanały stereo są identyczne. Brak clippingu pełnej skali nie wyklucza wcześniejszej kompresji/odszumiania. W tło (`R16`) wpisane są długie odcinki niemal ciszy. Odsłuch jest oryginalnej głośności, bez automatycznego wyrównania.

In [ ]:
RECORDING_ID='R06'
r=records.set_index('recording_id').loc[RECORDING_ID]
print(r.path)
y,sr=sf.read(OUT/'decoded'/f'{RECORDING_ID}.wav')
display(Audio(y,rate=sr,normalize=False));display(Image(filename=str(OUT/'figures'/f'{RECORDING_ID}.png')))
display(records[['recording_id','quiet_intervals']].query("recording_id in ['R16','R23','R24']"))

## 3. Segmentacja i tempo
Minimalny odstęp 250 ms z poprzedniego przykładu pomija szybkie trafienia. Tu używamy 120 ms. Reguły wstępne: kontrast ataku ≥12 dB, bez brzegów nagrania, pełne okno 20 ms przed i 150 ms po uderzeniu. Nie wymuszamy liczby trafień. Zdarzenia należy zatwierdzić odsłuchem — detektor wykrywa także inne impulsy.

Wykres jest klikalny: zaznacz fragment myszą, aby go przybliżyć. Kolory znaczą przyjęcie do cech, nie klasę OK/NOK.

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default='plotly_mimetype'
audit=pd.read_csv(OUT/'candidate_audit.csv'); e=audit[audit.recording_id==RECORDING_ID]
step=max(1,len(y)//6000);n=len(y)//step;b=y[:n*step].reshape(n,step);t=np.arange(n)*step/sr
fig=go.Figure([go.Scatter(x=t,y=b.min(1),mode='lines',line_color='#476675',showlegend=False),go.Scatter(x=t,y=b.max(1),mode='lines',line_color='#476675',showlegend=False)])
for hit in e.itertuples():fig.add_vline(x=hit.onset_s,line_color='#137f83' if hit.accepted else '#bda48c',line_width=1)
fig.update_layout(title=RECORDING_ID+' — kandydaci ataków',xaxis_title='Czas [s]',yaxis_title='Amplituda',height=350);fig.show()
display(pd.read_csv(OUT/'detector_sensitivity.csv').pivot(index='recording_id',columns='min_gap_s',values='n_candidates'))
display(e[['event_id','onset_s','attack_contrast_db','accepted','rejection']].head(12).round(3))

## 4. Dwa trafienia obok siebie
Zmień numery w EVENT_A/B. To porównanie kształtu widma i odsłuchu, bez nadawania trafieniu pewnej etykiety wady. Wybrane domyślnie trafienia są blisko środka czasowego danego nagrania.

In [ ]:
from pinglab.features import extract_features
from pinglab.plots import compare_hits
EVENT_A=events[events.recording_id=='R06'].iloc[len(events[events.recording_id=='R06'])//2].event_id
EVENT_B=events[events.recording_id=='R23'].iloc[len(events[events.recording_id=='R23'])//2].event_id
ya,sr=sf.read(OUT/'clips'/f'{EVENT_A}.wav');yb,_=sf.read(OUT/'clips'/f'{EVENT_B}.wav')
print(EVENT_A,EVENT_B)
display(Audio(ya,rate=sr,normalize=True));display(Audio(yb,rate=sr,normalize=True))
compare_hits([ya,yb],sr=sr,names=[EVENT_A,EVENT_B],offsets=[.02,.02]).show()

## 5. Widma, cechy i zmienność ustawienia
W każdej mapie punkt oznacza medianę cech nagrania; AST uśrednia 5 embeddingów. Zielony = OK, pomarańczowy = NOK, fioletowy = Bitten. Trójkąt = OLD, koło = pozostałe. PCA dopasowane do całej paczki nie mierzy skuteczności.

Physics: rezonans, środek i szerokość widma, roll-off, płaskość, entropia, zanik o 10 dB, energia późna/wczesna, liczba pików i sześć pasm. MFCC/mel opisują barwę. AST jest zamrożonym modelem AudioSet (115 zdarzeń, nie wszystkie); nie uczymy go na tej paczce.

In [ ]:
from client_recording_diagnostics import run as run_diagnostics
noise_summary=run_diagnostics(run_ast=RUN_AST)
display(Image(filename=str(OUT/'figures/spectral_conditions.png')))
for family in ['physics','mfcc','mel'] + (['ast'] if RUN_AST else []):
 display(Image(filename=str(OUT/'figures'/f'pca_{family}.png')))
display(pd.read_csv(OUT/'recording_summary.csv')[['recording_id','path','peak_hz','centroid_hz','decay10_s']].round(2))
display(pd.read_csv(OUT/'near_crack.csv').round(2))

## 6. Rzeczywiste tło: test wrażliwości cech
Dodajemy fragmenty tła z 19–30 s pliku R16, omijając długą ciszę. SNR dotyczy RMS całego okna. Test 115 zdarzeń pokazuje zmianę reprezentacji, nie accuracy. Skale physics/MFCC/mel różnią się, więc niższy słupek nie oznacza automatycznie lepszego klasyfikatora. Samo ściszenie o 12 dB jest kontrolą normalizacji.

Tło jest krótkie i prawdopodobnie przetworzone; ta symulacja nie zastępuje nagrań przy rzeczywistym hałasie stanowiska.

In [ ]:
display(noise_summary.round(3))
display(Image(filename=str(OUT/'figures/noise_features.png')))
yn,sr=sf.read(OUT/'decoded/R16.wav');display(Audio(yn,rate=sr,normalize=False))

## 7. Co możemy powiedzieć klientowi?
**Sygnał nadaje się do eksploracji. Najpilniejsze są porównywalne pomiary i metadane.**

- Uzupełnić ID fizycznych płytek w OLD i potwierdzić związek z Bad 1–2; Bitten trzymać osobno.
- Sprawdzić, czy nagrywanie usuwa tło: wyłączyć AGC/odszumianie/bramkę, nagrać próbę do WAV.
- Zachować identyczne podparcie, narzędzie, mikrofon i gain dla obu klas.
- 10–15 pojedynczych trafień w te same ponumerowane miejsca, co 1–2 s; osobny blok blisko pęknięcia.
- Dołożyć różne płytki OK, kolejne NOK w miarę dostępności, serię po ponownym ułożeniu i ciągły szum stanowiska.

Nie raportujemy accuracy/recall ani optymalnego progu. Nie znamy niezależności OK, a dwie pęknięte sztuki nie reprezentują różnorodności wad. Wizualne nakładanie się punktów nie dowodzi, że zadanie jest niemożliwe.

In [ ]:
from build_client_review import publish
publish()
assert not events.label.eq('background').any()
assert events.duration_s.between(.169,.171).all(), 'Cechy muszą mieć jednakowe okno 170 ms'
assert records.sha256.nunique()==len(records)
assert events.event_id.is_unique
from pinglab.features import feature_columns
assert np.isfinite(events[feature_columns(events,'physics')+feature_columns(events,'mfcc')+feature_columns(events,'mel')]).all().all()
print('Raport:',OUT/'START.html')
print('Kontrole: okna równe, brak tła w cechach, unikalne źródła i zdarzenia, wartości skończone.')